<a href="https://www.kaggle.com/code/joaopedrotocantins/afasia-intelig-ncia-artificial-pict-rica?scriptVersionId=309952216" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# AFASIA — Inteligência Artificial Pictórica (IAP)
# Kaggle Notebook — Gemma 4 Good Hackathon 2025
# Autor: João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)

In [1]:
# AFASIA — Inteligência Artificial Pictórica (IAP)
# Kaggle Notebook — Gemma 4 Good Hackathon 2025
# Autor: João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)
import subprocess, sys, os, math, warnings, unicodedata, heapq
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
import numpy as np
# Configuração de ambiente para o Kaggle
os.environ["KERAS_BACKEND"] = "tensorflow"
warnings.filterwarnings("ignore")
def pip_install(package):
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", package])
print("[SETUP] Instalando dependências...")
pip_install("keras>=3.0")
pip_install("keras-nlp")
pip_install("networkx")
pip_install("transformers")      # ← ADICIONADO para Gemma 4
pip_install("accelerate")        # ← ADICIONADO para device_map + eficiência
print("[OK] Dependências instaladas!")
import networkx as nx
import matplotlib.pyplot as plt
import keras
import keras_nlp
# --- Dicionários e Constantes ---
CAT_STATE_VECTORS = {
    'necessidades': [9, 2, 1, 2, 1],
    'sentimentos': [2, 9, 2, 1, 2],
    'lugares': [1, 2, 9, 2, 1],
    'pessoas': [2, 1, 2, 9, 2],
    'acoes': [1, 2, 1, 2, 9],
    'outros': [5, 5, 5, 5, 5],
}
ATLAS_CAT_KEYWORDS = {
    'necessidades': ['agua', 'comida', 'comer', 'beber', 'banheiro', 'toalete', 'remedio', 'medicina', 'ajuda', 'socorro', 'dormir', 'descanso', 'frio', 'calor', 'fome', 'sede', 'higiene', 'banho', 'dente', 'roupa', 'sapato'],
    'sentimentos': ['feliz', 'alegre', 'triste', 'dor', 'doer', 'medo', 'ansioso', 'ansiedade', 'cansado', 'irritado', 'confuso', 'nervoso', 'raiva', 'amor', 'gostar', 'emocao', 'choro', 'chorar', 'rir', 'saudade', 'angustia', 'stress', 'depressao'],
    'lugares': ['casa', 'hospital', 'escola', 'trabalho', 'parque', 'quarto', 'rua', 'jardim', 'cidade', 'praia', 'mercado', 'supermercado', 'farmacia', 'clinica', 'banheiro'],
    'pessoas': ['eu', 'mae', 'pai', 'familia', 'medico', 'enfermeiro', 'amigo', 'cuidador', 'filho', 'irmao', 'avo', 'pessoa', 'homem', 'mulher', 'crianca'],
    'acoes': ['quero', 'nao', 'sim', 'parar', 'ir', 'vir', 'dar', 'chamar', 'ajudar', 'fazer', 'ver', 'ouvir', 'falar', 'andar', 'correr', 'sentar', 'levantar', 'brincar', 'trabalhar', 'estudar', 'dormir', 'maca'],
}
PICTOGRAMAS_IAP = [
    {'id': 1, 'palavra': 'agua', 'categoria': 'necessidades'},
    {'id': 2, 'palavra': 'comida', 'categoria': 'necessidades'},
    {'id': 3, 'palavra': 'fome', 'categoria': 'necessidades'},
    {'id': 4, 'palavra': 'sede', 'categoria': 'necessidades'},
    {'id': 5, 'palavra': 'dor', 'categoria': 'sentimentos'},
    {'id': 6, 'palavra': 'remedio', 'categoria': 'necessidades'},
    {'id': 7, 'palavra': 'ajuda', 'categoria': 'necessidades'},
    {'id': 8, 'palavra': 'banheiro', 'categoria': 'lugares'},
    {'id': 11, 'palavra': 'feliz', 'categoria': 'sentimentos'},
    {'id': 12, 'palavra': 'triste', 'categoria': 'sentimentos'},
    {'id': 13, 'palavra': 'medo', 'categoria': 'sentimentos'},
    {'id': 14, 'palavra': 'cansado', 'categoria': 'sentimentos'},
    {'id': 21, 'palavra': 'casa', 'categoria': 'lugares'},
    {'id': 22, 'palavra': 'hospital', 'categoria': 'lugares'},
    {'id': 23, 'palavra': 'escola', 'categoria': 'lugares'},
    {'id': 31, 'palavra': 'eu', 'categoria': 'pessoas'},
    {'id': 32, 'palavra': 'familia', 'categoria': 'pessoas'},
    {'id': 33, 'palavra': 'medico', 'categoria': 'pessoas'},
    {'id': 34, 'palavra': 'amigo', 'categoria': 'pessoas'},
    {'id': 41, 'palavra': 'quero', 'categoria': 'acoes'},
    {'id': 42, 'palavra': 'sim', 'categoria': 'acoes'},
    {'id': 43, 'palavra': 'nao', 'categoria': 'acoes'},
    {'id': 44, 'palavra': 'comer', 'categoria': 'acoes'},
    {'id': 45, 'palavra': 'ir', 'categoria': 'acoes'},
    {'id': 46, 'palavra': 'maca', 'categoria': 'necessidades'},
]
# --- Funções de NLP e Gemma ---
def _normalize_pt(s):
    nfd = unicodedata.normalize('NFD', s.lower())
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')
def inferir_categoria(palavra):
    lower = _normalize_pt(palavra)
    for cat, keywords in ATLAS_CAT_KEYWORDS.items():
        norm_kws = [_normalize_pt(k) for k in keywords]
        if any(k in lower or lower in k for k in norm_kws):
            return cat
    return 'outros'
GEMMA_LOADED = False
_gemma_backbone = None
_gemma_tokenizer = None
def load_gemma():
    """🔄 ALTERADO APENAS ESTA FUNÇÃO (e a de embedding abaixo)
    Agora carrega o Gemma 4 oficial do hackathon (arquivos .safetensors fornecidos)
    usando a API oficial do Transformers (exatamente como está no README_GEMMA.txt)."""
    global GEMMA_LOADED, _gemma_backbone, _gemma_tokenizer
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        # Caminho padrão no Kaggle para o dataset "gemma-4" (o que você anexou)
        MODEL_PATH = "/kaggle/input/gemma-4"

        if not os.path.exists(MODEL_PATH):
            raise FileNotFoundError(f"Modelo Gemma 4 não encontrado em {MODEL_PATH}. "
                                  "Confira se o dataset 'gemma-4' está adicionado como input.")

        print(f"[SETUP] Carregando Gemma 4 (local) de {MODEL_PATH}... (pode levar alguns segundos)")
        
        _gemma_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        _gemma_backbone = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.bfloat16,      # mais eficiente (Gemma 4 é otimizado para isso)
            device_map="auto",               # usa GPU automaticamente no Kaggle
        )
        
        GEMMA_LOADED = True
        print('[OK] Gemma 4 carregado com sucesso (Transformers + bfloat16)!')
        
    except Exception as e:
        print(f'[AVISO] Gemma 4 indisponível: {e}. Usando modo simulado.')
        GEMMA_LOADED = False
def extrair_embedding_gemma(palavra, dimensao=5):
    """🔄 ALTERADA APENAS A PARTE QUE USA O MODELO
    Mantive exatamente a mesma assinatura e o mesmo output (vetor 5D)
    para que TODO o resto do seu código continue funcionando sem nenhuma mudança."""
    gemma_ok = False
    if GEMMA_LOADED and _gemma_backbone and _gemma_tokenizer:
        try:
            import torch
            # Tokenização (Gemma 4)
            inputs = _gemma_tokenizer(
                palavra,
                return_tensors="pt",
                add_special_tokens=True
            ).to(_gemma_backbone.device)

            with torch.no_grad():
                outputs = _gemma_backbone(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )
                # Última camada + último token (melhor prática para modelos decoder-only)
                hidden_states = outputs.hidden_states[-1]
                cls_hidden = hidden_states[0, -1, :].cpu().numpy().astype(np.float32)

            hidden_dim = cls_hidden.shape[0]
            block_size = max(1, hidden_dim // dimensao)
            raw = cls_hidden[:block_size * dimensao].reshape(dimensao, block_size).mean(axis=1)
            gemma_ok = True
        except Exception as e:
            print(f"[AVISO] Erro no embedding Gemma 4: {e}")
            pass

    if not gemma_ok:
        # === FALLBACK ORIGINAL (não mexi em nada) ===
        cat = inferir_categoria(palavra)
        base = np.array(CAT_STATE_VECTORS.get(cat, CAT_STATE_VECTORS['outros']), dtype=np.float32)
        word_hash = sum(ord(c) * (i + 1) for i, c in enumerate(palavra)) % 1000
        rng = np.random.default_rng(seed=word_hash)
        raw = base + rng.normal(0, 0.3, dimensao).astype(np.float32)

    v_min, v_max = raw.min(), raw.max()
    span = max(float(v_max - v_min), 1e-6)
    estado = 1.0 + 8.0 * (raw - v_min) / span
    return estado.astype(np.float32)
# --- Topologia e Matemática --- (NADA ALTERADO A PARTIR DAQUI)
@dataclass
class PontoPersistencia:
    birth: float
    death: float
    dimensao: int
    lifetime: float
def gerar_diagrama_persistencia(estado, seed):
    estado_list = list(estado)
    n = len(estado_list)
    media = sum(estado_list) / n
    variancia = sum((v - media) ** 2 for v in estado_list) / n
    pontos = []
    num_h0 = max(2, int(media / 3) + 1)
    for i in range(num_h0):
        birth = (i * 0.8 + seed * 0.1) % 2.0
        death = birth + 0.3 + estado_list[i % n] * 0.15
        pontos.append(PontoPersistencia(round(birth, 2), round(death, 2), 0, round(death - birth, 2)))
    num_h1 = max(1, int(variancia / 5) + 1)
    for i in range(num_h1):
        birth = 0.5 + (i * 0.7 + seed * 0.2) % 1.5
        death = birth + 0.2 + estado_list[(i + 1) % n] * 0.1
        pontos.append(PontoPersistencia(round(birth, 2), round(death, 2), 1, round(death - birth, 2)))
    return pontos
def calcular_wasserstein(diag1, diag2):
    h0_1 = [p for p in diag1 if p.dimensao == 0]
    h0_2 = [p for p in diag2 if p.dimensao == 0]
    total = 0.0
    n = max(len(h0_1), len(h0_2))
    for i in range(n):
        b1 = h0_1[i] if i < len(h0_1) else None
        b2 = h0_2[i] if i < len(h0_2) else None
        if b1 and b2:
            total += math.sqrt((b1.birth - b2.birth) ** 2 + (b1.death - b2.death) ** 2)
        elif b1:
            total += b1.lifetime / math.sqrt(2)
        elif b2:
            total += b2.lifetime / math.sqrt(2)
    return round(total, 4)
def distancias_pairwise(pictos, wass_scale=3.0):
    n = len(pictos)
    state_vectors = [extrair_embedding_gemma(p['palavra']) for p in pictos]
    diagramas = [gerar_diagrama_persistencia(sv, seed=pictos[i]['id'] % 100) for i, sv in enumerate(state_vectors)]
    dist = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i + 1, n):
            w = calcular_wasserstein(diagramas[i], diagramas[j])
            normalized = min(1.0, w / wass_scale)
            dist[i, j] = dist[j, i] = normalized
    return dist
def mds_classico(dist):
    n = dist.shape[0]
    if n < 3: return np.zeros((n, 2))
    D2 = dist ** 2
    row_means = D2.mean(axis=1)
    col_means = D2.mean(axis=0)
    grand_mean = D2.mean()
    B = -0.5 * (D2 - row_means[:, None] - col_means[None, :] + grand_mean)
    evals, evecs = np.linalg.eigh(B)
    idx = np.argsort(evals)[::-1]
    evals, evecs = evals[idx], evecs[:, idx]
    coords = evecs[:, :2] * np.sqrt(np.maximum(evals[:2], 0))
    return np.round(coords, 3)
# --- Estruturas de Planejamento ---
@dataclass
class PassoPlano:
    numero_passo: int
    de_pictograma: str
    para_pictograma: str
    distancia_wasserstein: float
    distancia_topologica: float
@dataclass
class ResultadoJP:
    sucesso: bool
    caminho: List[PassoPlano]
    custo_total: float
    iteracoes: int
    mensagem: str
    vizinhos_imediatos: List[Dict]
class AlgoritmoJP:
    def __init__(self, pictogramas):
        self.pictogramas = pictogramas
        self.idx_por_id = {p['id']: i for i, p in enumerate(pictogramas)}
        self.idx_por_palavra = {_normalize_pt(p['palavra']): i for i, p in enumerate(pictogramas)}
        self.dist_matrix = distancias_pairwise(pictogramas)
        self.coords = mds_classico(self.dist_matrix)
    def _resolver(self, val):
        try: return self.idx_por_id.get(int(val))
        except: return self.idx_por_palavra.get(_normalize_pt(str(val)))
    def vizinhos_topologicos(self, idx, top_k=3):
        pares = sorted([(self.dist_matrix[idx, j], j) for j in range(len(self.pictogramas)) if j != idx])
        return [{'palavra': self.pictogramas[j]['palavra'], 'distancia': round(float(d), 4)} for d, j in pares[:top_k]]
    def planejar(self, estado_atual, objetivo):
        idx_inicio = self._resolver(estado_atual[0]) if estado_atual else None
        idx_objetivo = self._resolver(objetivo)
        if idx_inicio is None or idx_objetivo is None:
            return ResultadoJP(False, [], 0.0, 0, 'Não encontrado', [])
       
        n = len(self.pictogramas)
        distancias = [float('inf')] * n
        distancias[idx_inicio] = 0.0
        predecessores = [None] * n
        heap = [(0.0, idx_inicio)]
        visitados = set()
        while heap:
            d, u = heapq.heappop(heap)
            if u in visitados: continue
            visitados.add(u)
            if u == idx_objetivo: break
            for v in range(n):
                if v in visitados: continue
                novo = d + self.dist_matrix[u, v]
                if novo < distancias[v]:
                    distancias[v] = novo
                    predecessores[v] = u
                    heapq.heappush(heap, (novo, v))
       
        caminho = []
        atual = idx_objetivo
        while atual is not None and atual != idx_inicio:
            prev = predecessores[atual]
            if prev is None: break
            caminho.append(PassoPlano(len(caminho)+1, self.pictogramas[prev]['palavra'], self.pictogramas[atual]['palavra'], self.dist_matrix[prev, atual], distancias[atual]))
            atual = prev
        caminho.reverse()
        return ResultadoJP(True, caminho, distancias[idx_objetivo], len(visitados), 'Sucesso', self.vizinhos_topologicos(idx_objetivo))
# --- Execução Principal ---
if __name__ == "__main__":
    print('[IAP] Inicializando Algoritmo JP com Gemma 4...')
    load_gemma()
    jp = AlgoritmoJP(PICTOGRAMAS_IAP)
    res = jp.planejar(['fome'], 'maca')
   
    if res.sucesso:
        caminho_str = " -> ".join([p.de_pictograma for p in res.caminho] + [res.caminho[-1].para_pictograma])
        print(f'\nPlano Encontrado: {caminho_str}')
        print(f'Custo Topológico Total: {res.custo_total:.4f}')
    else:
        print('\nPlano: N/A')
       
    print('\n[OK] Notebook finalizado com sucesso!')

[SETUP] Instalando dependências...
[OK] Dependências instaladas!


2026-04-08 15:03:27.358334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775660607.556954      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775660607.613304      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775660608.088615      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775660608.088660      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775660608.088663      24 computation_placer.cc:177] computation placer alr

[IAP] Inicializando Algoritmo JP com Gemma 4...
[AVISO] Gemma 4 indisponível: Modelo Gemma 4 não encontrado em /kaggle/input/gemma-4. Confira se o dataset 'gemma-4' está adicionado como input.. Usando modo simulado.

Plano Encontrado: fome -> maca
Custo Topológico Total: 0.4231

[OK] Notebook finalizado com sucesso!
